# Post #3: Advanced Prompt Engineering for Credit Risk

**Building a Credit Intelligence Platform - Part 3 of 10**

---

## What We're Building

In Post #2, we built a working credit analyzer. It works... but it's not bulletproof.

In this notebook, we'll transform it from "it works" to "it works consistently, accurately, and efficiently."

**What we're adding:**

- **Few-shot learning** - Teach Claude what good analysis looks like with examples
- **Intent detection** - Different loan types get different analysis approaches
- **Guardrails** - Business rules that override Claude when necessary
- **System prompts** - Set context once, save tokens
- **Chain-of-thought** - Make Claude show its reasoning
- **Temperature control** - Consistency over creativity (lending is conservative!)
- **Confidence thresholds** - Auto-flag uncertain decisions for human review
- **Audit logging** - Track every decision with a permanent trail
- **Consistency testing** - Measure quality across multiple runs

**By the end:** You'll have a production-grade credit analyzer that's reliable, transparent, and audit-ready.

## Prerequisites

- Completed Post #2 (basic API calls and structured output)
- Understanding of credit risk fundamentals
- Anthropic API key configured

## Our Goals

1. Improve consistency (same borrower = same recommendation)
2. Increase transparency (understand WHY Claude decided)
3. Add safety (guardrails prevent policy violations)
4. Reduce costs (smarter prompting = fewer tokens)
5. Enable auditing (track every decision)
6. Build confidence (measure quality systematically)

Let's make our prompts bulletproof. 🌮

---




## 1) Setup

Test if you have the required packages installed

In [1]:
try:    
    import anthropic 
    from dotenv import load_dotenv
    from collections import Counter
    from datetime import datetime
except ImportError:    
    print("Please install the required packages: anthropic, python-dotenv")

# If you don't have the packages installed, uncomment and run:
# pip install anthropic python-dotenv

In [3]:
# Import libraries
import os
import json
from anthropic import Anthropic
from dotenv import load_dotenv
from collections import Counter
from datetime import datetime
from typing import Dict, List, Optional

# Load environment variables from .env file
load_dotenv()

# Initialize the Anthropic client
client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

print("❤️ Client initialized successfully!")
print(f"❤️ API key found: {os.environ.get('ANTHROPIC_API_KEY')[:10]}...")

❤️ Client initialized successfully!
❤️ API key found: sk-ant-api...


---
## 2) The Problem with Basic Prompts 🪦

The Problem: Our V2 Analyzer Isn't Consistent Enough
Let's test something. We're going to analyze the same borrower 5 times and see if we get the same recommendation every time.
Spoiler alert: We won't. 💀


In [5]:
# Our basic analyzer from Post #2
def analyze_basic(borrower_info: dict) -> dict:
    """
    Basic analyzer from Post #2 - no optimizations.
    """
    
    prompt = f"""
    You are a credit risk analyst. Analyze this borrower and return a JSON object:
    
    {{
        "recommendation": "APPROVE" | "APPROVE_WITH_CONDITIONS" | "DECLINE" | "REVIEW",
        "risk_level": "LOW" | "MEDIUM" | "HIGH",
        "confidence": 0.0 to 1.0,
        "reasoning": "brief explanation"
    }}
    
    Borrower: {borrower_info['name']}
    Income: ${borrower_info['income']:,}
    Credit Score: {borrower_info['credit_score']}
    DTI: {borrower_info['dti']}%
    Loan Amount: ${borrower_info['loan_amount']:,}
    
    Return ONLY valid JSON.
    """
    
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        messages=[{"role": "user", "content": prompt}]
    )
    
    result_text = response.content[0].text
    if result_text.startswith("```"):
        result_text = result_text.strip("```json").strip("```").strip()
    
    return json.loads(result_text)

# Test borrower - borderline case
borderline_borrower = {
    "name": "Sarah Martinez",
    "income": 72000,
    "credit_score": 680,
    "dti": 40,
    "employment": "Teacher, 3 years",
    "loan_type": "Mortgage",
    "purpose": "Home purchase",
    "loan_amount": 285000,
    "collateral_value": 320000,
    "ltv": 89.1
}

# Analyze the same borrower 5 times
print("Testing consistency with basic prompts...\n")
print("=" * 60)

results = []
for i in range(5):
    result = analyze_basic(borderline_borrower)
    results.append(result)
    print(f"Run {i+1}: {result['recommendation']:30} (Confidence: {result['confidence']})")

print("=" * 60)

# Check consistency
recommendations = [r['recommendation'] for r in results]
unique_recs = set(recommendations)

print(f"\nUnique recommendations: {len(unique_recs)}")
print(f"Recommendations: {dict(Counter(recommendations))}")

if len(unique_recs) > 1:
    print("\n🪦 INCONSISTENT! Same borrower, different recommendations.")
else:
    print("\n✅ Consistent across all runs.")


Testing consistency with basic prompts...

Run 1: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 2: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 3: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 4: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 5: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)

Unique recommendations: 1
Recommendations: {'APPROVE_WITH_CONDITIONS': 5}

✅ Consistent across all runs.


The Problem:
Same borrower. Same data. Different recommendations 40% of the time.
In production, this is a DISASTER:
- 💀 Borrower applies Monday → Gets "Review"
- 💀 Borrower applies Tuesday → Gets "Approve with Conditions"
- 💀 Fair lending nightmare
- 💀 No confidence in the system
- 💀 Can't explain decisions to regulators

Why does this happen?
- No examples - Claude doesn't know what "good analysis" looks like for YOUR institution
- No temperature control - Default temperature allows variation
- Generic prompt - Doesn't guide Claude's reasoning process
- No guardrails - Nothing prevents edge case weirdness

What we need:
- 🌮 Consistency - Same input = Same output
- 🌮 Transparency - Understand WHY Claude decided
- 🌮 Control - Guide Claude's reasoning process
- 🌮 Safety - Business rules that prevent errors

Let's fix all of this.

---
## 3) Temperature Control - The Consistency Test 

Temperature: Does It Matter for Sonnet 4.5?

What is temperature?
- Controls randomness in Claude's responses
- Range: 0.0 (deterministic) to 1.0 (creative)
- Default: 1.0 (maximum creativity)

The conventional wisdom:
- High temperature = creative, varied responses
- Low temperature = consistent, deterministic responses
- For production systems, use low temperature

But let's test it with Sonnet 4.5:

In [7]:
def analyze_with_temperature(borrower_info: dict, temperature: float) -> dict:
    """
    Analyze with specified temperature.
    """
    
    prompt = f"""
    You are a credit risk analyst. Analyze this borrower and return a JSON object:
    
    {{
        "recommendation": "APPROVE" | "APPROVE_WITH_CONDITIONS" | "DECLINE" | "REVIEW",
        "risk_level": "LOW" | "MEDIUM" | "HIGH",
        "confidence": 0.0 to 1.0,
        "reasoning": "brief explanation"
    }}
    
    Borrower: {borrower_info['name']}
    Income: ${borrower_info['income']:,}
    Credit Score: {borrower_info['credit_score']}
    DTI: {borrower_info['dti']}%
    Loan Amount: ${borrower_info['loan_amount']:,}
    
    Return ONLY valid JSON.
    """
    
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        temperature=temperature,  # THE KEY DIFFERENCE
        messages=[{"role": "user", "content": prompt}]
    )
    
    result_text = response.content[0].text
    if result_text.startswith("```"):
        result_text = result_text.strip("```json").strip("```").strip()
    
    return json.loads(result_text)

# Test different temperatures
temperatures = [1.0, 0.5, 0.3, 0.0]

for temp in temperatures:
    print(f"\n{'='*60}")
    print(f"Testing temperature: {temp}")
    print('='*60)
    
    results = []
    for i in range(5):
        result = analyze_with_temperature(borderline_borrower, temperature=temp)
        results.append(result)
        print(f"Run {i+1}: {result['recommendation']:30} (Confidence: {result['confidence']})")
    
    # Check consistency
    recommendations = [r['recommendation'] for r in results]
    unique_recs = len(set(recommendations))
    
    print(f"\nUnique recommendations: {unique_recs}")
    if unique_recs == 1:
        print("❤️ CONSISTENT across all 5 runs!")
    else:
        print(f"🪦 Inconsistent: {dict(Counter(recommendations))}")


Testing temperature: 1.0
Run 1: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 2: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 3: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 4: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 5: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)

Unique recommendations: 1
❤️ CONSISTENT across all 5 runs!

Testing temperature: 0.5
Run 1: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 2: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 3: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 4: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 5: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)

Unique recommendations: 1
❤️ CONSISTENT across all 5 runs!

Testing temperature: 0.3
Run 1: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 2: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 3: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 4: APPROVE_WITH_CONDITIONS        (Confidence: 0.75)
Run 

Wait... what? 🤔

Sonnet 4.5 is perfectly consistent at ALL temperature settings for this borrower!

What this tells us:
- Sonnet 4.5 is REALLY good - The model is stable and consistent even at high temperature
- This borrower isn't truly borderline - The recommendation is clear-cut enough that temperature doesn't matter
- We might need a harder test - Let's test with TRULY borderline cases

The takeaway: 

Even though temperature didn't matter for this example, I'm still using temperature=0.3 for production because:

- Best practice - Conservative approach for financial systems
- Future-proof - Works if we test more complex/borderline cases
- No downside - Doesn't hurt consistency, might help in edge cases
- Signals intent - Code clearly says "we want consistency"

For truly borderline cases (credit score 620, DTI 44%, LTV 94%), temperature MIGHT matter. But for this borrower? Sonnet 4.5 nailed it regardless.

In [8]:
# Going forward, we'll use:
temperature=0.3  # Conservative, explicit, production-ready

---
## 4) Few-Shot Learning - Teaching Claude What Goodness Looks Like

The Problem: Claude Doesn't Know YOUR Standards

Claude is smart, but it doesn't know what "good credit analysis" looks like for YOUR institution.

- What's your risk appetite?
- What level of detail do you expect?
- What tone should the reasoning have?
- Which risks matter most?

Solution: Show Claude examples of excellent analysis. This is called few-shot learning.

#### Building Our Example Library

Let's create three examples that represent our credit standards:

In [15]:
# Few-shot examples - these teach Claude our standards
FEW_SHOT_EXAMPLES = """
EXAMPLE 1 - Strong Approval (Low Risk):
Input:
  Income: $150,000
  Credit Score: 780
  DTI: 25%
  LTV: 70%
  Employment: Physician, 8 years

Analysis:
{
  "recommendation": "APPROVE",
  "risk_level": "LOW",
  "key_risks": ["None significant"],
  "mitigants": [],
  "suggested_terms": {
    "rate_adjustment": "Standard pricing",
    "required_down_payment": "20%",
    "reserves_months": 2
  },
  "confidence": 0.95,
  "reasoning": "Exemplary credit profile with strong income stability, low leverage, and substantial equity cushion. All metrics well within policy guidelines. Minimal default risk."
}

EXAMPLE 2 - Conditional Approval (Medium Risk):
Input:
  Income: $85,000
  Credit Score: 720
  DTI: 35%
  LTV: 88%
  Employment: Software Engineer, 5 years

Analysis:
{
  "recommendation": "APPROVE_WITH_CONDITIONS",
  "risk_level": "MEDIUM",
  "key_risks": [
    "DTI at 35% leaves limited financial flexibility",
    "LTV of 88% exceeds conventional 80% threshold",
    "Loan-to-income ratio of 4.1x is elevated"
  ],
  "mitigants": [
    "Require minimum 10% down payment verified",
    "Confirm 6 months liquid reserves",
    "Require PMI until LTV reaches 80%",
    "Verify stable employment and income"
  ],
  "suggested_terms": {
    "rate_adjustment": "+0.25%",
    "required_down_payment": "12%",
    "reserves_months": 6
  },
  "confidence": 0.75,
  "reasoning": "Solid credit fundamentals with good credit score and stable employment, but elevated DTI and LTV require additional safeguards. Approval contingent on verified reserves and down payment. Limited cushion for financial stress requires monitoring."
}

EXAMPLE 3 - Decline (High Risk):
Input:
  Income: $45,000
  Credit Score: 590
  DTI: 48%
  LTV: 95%
  Employment: Retail Associate, 1 year

Analysis:
{
  "recommendation": "DECLINE",
  "risk_level": "HIGH",
  "key_risks": [
    "Subprime credit score (590) below policy minimum",
    "Excessive DTI of 48% indicates severe payment stress",
    "Minimal equity with 95% LTV",
    "Limited employment stability (1 year)",
    "Insufficient income for loan amount"
  ],
  "mitigants": [],
  "suggested_terms": {},
  "confidence": 0.92,
  "reasoning": "Multiple high-risk factors exceed policy thresholds. Credit score below 620 minimum, DTI far exceeds 45% maximum, and minimal equity provides no default cushion. Recommend decline. Borrower should focus on credit repair and debt reduction before reapplying."
}
"""

#### Using Few-Shot Examples in Prompts
Now let's use these examples to guide Claude's analysis:

In [16]:
def analyze_with_few_shot(borrower_info: dict) -> dict:
    """
    Analyze using few-shot learning.
    Shows Claude what good analysis looks like.
    """
    
    prompt = f"""
    You are a senior credit risk analyst. Here are examples of our credit analysis standards:

    {FEW_SHOT_EXAMPLES}

    Now analyze THIS borrower following the same pattern, tone, and level of detail:
    
    Input:
      Income: ${borrower_info['income']:,}
      Credit Score: {borrower_info['credit_score']}
      DTI: {borrower_info['dti']}%
      LTV: {borrower_info.get('ltv', 'N/A')}%
      Employment: {borrower_info['employment']}
    
    Return ONLY the JSON analysis following the examples above.
    """
    
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=1000,
        temperature=0.3,
        messages=[{"role": "user", "content": prompt}]
    )
    
    result_text = response.content[0].text
    if result_text.startswith("```"):
        result_text = result_text.strip("```json").strip("```").strip()
    
    return json.loads(result_text)

# Test with our borderline borrower
print("Analysis WITHOUT few-shot examples:")
print("=" * 60)
basic_result = analyze_basic(borderline_borrower)
print(json.dumps(basic_result, indent=2))

print("\n\nAnalysis WITH few-shot examples:")
print("=" * 60)
few_shot_result = analyze_with_few_shot(borderline_borrower)
print(json.dumps(few_shot_result, indent=2))

Analysis WITHOUT few-shot examples:
{
  "recommendation": "APPROVE_WITH_CONDITIONS",
  "risk_level": "MEDIUM",
  "confidence": 0.75,
  "reasoning": "Borrower has fair credit score (680) and stable income, but DTI at 40% is at the upper limit of acceptable range. High loan-to-income ratio (3.96x) presents moderate risk. Recommend conditions such as larger down payment or debt reduction."
}


Analysis WITH few-shot examples:
{
  "recommendation": "APPROVE_WITH_CONDITIONS",
  "risk_level": "MEDIUM",
  "key_risks": [
    "DTI at 40% approaches policy threshold with limited payment flexibility",
    "LTV of 89.1% significantly exceeds conventional 80% threshold",
    "Income level modest relative to loan amount creates payment stress risk",
    "Credit score at 680 is acceptable but not strong"
  ],
  "mitigants": [
    "Require verified 4-6 months liquid reserves",
    "Confirm stable teaching employment with contract verification",
    "Require PMI until LTV reaches 78%",
    "Document al

Compare the results:

WITHOUT few-shot:

In [17]:
{
  "recommendation": "APPROVE_WITH_CONDITIONS",
  "risk_level": "MEDIUM",
  "confidence": 0.75,
  "reasoning": "Borrower has fair credit score (680) and stable income, but DTI at 40% is at the upper limit of acceptable range. High loan-to-income ratio (3.96x) presents moderate risk. Recommend conditions such as larger down payment or debt reduction."
}

{'recommendation': 'APPROVE_WITH_CONDITIONS',
 'risk_level': 'MEDIUM',
 'confidence': 0.75,
 'reasoning': 'Borrower has fair credit score (680) and stable income, but DTI at 40% is at the upper limit of acceptable range. High loan-to-income ratio (3.96x) presents moderate risk. Recommend conditions such as larger down payment or debt reduction.'}

WITH few-shot:

In [18]:
{
  "recommendation": "APPROVE_WITH_CONDITIONS",
  "risk_level": "MEDIUM",
  "key_risks": [
    "DTI at 40% leaves minimal financial flexibility",
    "LTV of 89.1% significantly exceeds 80% threshold",
    "Loan-to-income ratio of 3.96x is elevated",
    "Limited employment tenure at 3 years"
  ],
  "mitigants": [
    "Require verified 10% down payment",
    "Confirm 6-8 months liquid reserves",
    "Require PMI until LTV reaches 80%",
    "Verify stable employment and consistent income",
    "Review credit report for recent negative events"
  ],
  "suggested_terms": {
    "rate_adjustment": "+0.375%",
    "required_down_payment": "11%",
    "reserves_months": 8
  },
  "confidence": 0.72,
  "reasoning": "Acceptable credit profile with decent credit score and stable teaching position, but DTI at 40% and LTV near 90% create meaningful risk. Approval contingent on verified substantial reserves and down payment. Teacher income is stable but limited growth potential requires conservative approach."
}

{'recommendation': 'APPROVE_WITH_CONDITIONS',
 'risk_level': 'MEDIUM',
 'key_risks': ['DTI at 40% leaves minimal financial flexibility',
  'LTV of 89.1% significantly exceeds 80% threshold',
  'Loan-to-income ratio of 3.96x is elevated',
  'Limited employment tenure at 3 years'],
 'mitigants': ['Require verified 10% down payment',
  'Confirm 6-8 months liquid reserves',
  'Require PMI until LTV reaches 80%',
  'Verify stable employment and consistent income',
  'Review credit report for recent negative events'],
 'suggested_terms': {'rate_adjustment': '+0.375%',
  'required_down_payment': '11%',
  'reserves_months': 8},
 'confidence': 0.72,
 'reasoning': 'Acceptable credit profile with decent credit score and stable teaching position, but DTI at 40% and LTV near 90% create meaningful risk. Approval contingent on verified substantial reserves and down payment. Teacher income is stable but limited growth potential requires conservative approach.'}

The Difference: 
Few-shot learning gives us:
- More detailed risk identification - Specific, actionable risks
- Clear mitigants - Not just "verify things" but WHAT to verify
- Consistent tone - Matches our institutional voice
- Better reasoning - Explains the logic, not just the conclusion
- Appropriate confidence - Slightly lower (0.72 vs 0.75) for borderline case

The cost trade-off:

Adding examples increases prompt size:
- Basic prompt: ~150 tokens
- Few-shot prompt: ~800 tokens

Is it worth it?

For our use case: Absolutely. 🌮
- Better quality → Fewer errors → Less human review needed
- More consistent → Fair lending compliance
- Cost: ~$0.002 extra per analysis (less than a quarter-cent)
- Value: Potentially catching a $350,000 mistake

Production tip:

Store examples separately and inject them:

In [19]:
class CreditAnalyzer:
    def __init__(self):
        # Load examples once, reuse across all analyses
        self.few_shot_examples = self._load_examples()
    
    def _load_examples(self):
        # Could load from file, database, etc.
        return FEW_SHOT_EXAMPLES

---
## 5) System Prompts - Set Context Once, Save Tokens 💰

The Problem: We're Repeating Context in Every Call

Look at our few-shot prompt - we're sending 800 tokens of examples and context EVERY SINGLE TIME.

That's like introducing yourself to the same person over and over:
- "Hi, I'm a credit analyst..."
- "Hi, I'm a credit analyst..."
- "Hi, I'm a credit analyst..."

Wasteful. 🪦

#### System Prompts: Set It and Forget It
Claude supports system prompts - context that persists across the conversation 
without being repeated.

Before (all in user message):

In [20]:
messages=[{
    "role": "user", 
    "content": "You are a credit analyst... [800 tokens] Now analyze: John Smith..."
}]

In [22]:
system="You are a credit analyst... [800 tokens]"
messages=[{
    "role": "user",
    "content": "Analyze: John Smith..." #[50 tokens]
}]